# Shape step walkthrough: one render, one photo

This notebook sends two images through the shape step of our pipeline and shows
a picture after every step:

1. find the brick (YOLO mask, `brick_mask` where YOLO finds nothing)
2. cut out a grey 96 × 96 crop and stretch its contrast
3. augmentation (only during training)
4. predict the part number (top 3)

Every step calls the real code in `src/legorec/`. One plot (the histogram in
step 2) recomputes an intermediate result to show what happens inside a
function.

**Before you run it**

- Install: `pip install -e ".[train,docs]" ipykernel`, then pick the `.venv`
  kernel (top right in VS Code).
- Models in `models/`: `shape.pt` (the team default, see the README) and
  `bricks_yolo.pt` (the brick finder).
- Two images in `data/raw/examples/` (this folder is not in Git): a render and a
  photo of a single brick. Or change the paths below.

Unknown words (render, top-1, …)? See the [glossary in the README](../README.md#glossary).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from legorec.paths import MODELS_DIR, REPO_ROOT
from legorec.segment.bricks import single_brick_masks
from legorec.shape.augment import augment_batch, denormalize, normalize
from legorec.shape.model import ShapeClassifier
from legorec.shape.preprocess import brick_mask, preprocess_crop
from legorec.types import BrickCrop

# Inputs: change these paths to try other images.
EXAMPLES = REPO_ROOT / "data" / "raw" / "examples"
IMAGE_PATHS = {
    "render": EXAMPLES / "3002 brick 2x3 011L.png",
    "photo": EXAMPLES / "3002_red_catalog_image.jpeg",
}
TRUE_PART = "3002"  # both images show a brick 2x3

MODEL_PATH = MODELS_DIR / "shape.pt"
FIGURES = REPO_ROOT / "docs" / "figures"  # generated, not in Git


def hide_ticks(ax):
    """No axis numbers, but keep the frame (a white photo stays visible)."""
    ax.set_xticks([])
    ax.set_yticks([])


def show(pictures, titles, size=3.5):
    """Draw images next to each other. Grey images are shown without automatic
    contrast: 0 is black, 255 is white."""
    _, axes = plt.subplots(1, len(pictures), figsize=(size * len(pictures), size))
    for ax, picture, title in zip(np.atleast_1d(axes), pictures, titles):
        ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        hide_ticks(ax)
    plt.show()

## Step 0: the two inputs

- **Render** (left): a computer-generated image, 400 × 400 px, grey brick on a
  black background (an old render from the Kaggle set; any render of one brick
  works, e.g. from `data/raw/internet/renders/`).
- **Photo** (right): a catalog-style image of a red brick on a white
  background, not one of our own photos. Replace it with one of your own photos.

Both show the same part: **3002, brick 2×3**.

In [ ]:
images = {name: Image.open(path).convert("RGB") for name, path in IMAGE_PATHS.items()}

show(
    [np.asarray(image) for image in images.values()],
    [f"{name}: {image.width} x {image.height} px" for name, image in images.items()],
)

## Step 1: find the brick

The same YOLO model that finds the bricks on our sheet photos also finds the
brick here (`single_brick_masks`). The image is first scaled so the brick has
about the size it has on a sheet, because that is what YOLO was trained on.
Training images go the same way, so the model learns from crops that look like
the ones it gets later.

Where YOLO finds nothing (often a white brick on a light background),
`brick_mask` takes over: everything whose colour differs clearly from the
image border is brick, and only the largest region counts.

In [ ]:
rgbs = {name: np.asarray(image) for name, image in images.items()}
masks, mask_source = {}, {}
for (name, rgb), yolo_mask in zip(
    rgbs.items(), single_brick_masks(list(rgbs.values()))
):
    masks[name] = yolo_mask if yolo_mask is not None else brick_mask(rgb)
    mask_source[name] = "YOLO" if yolo_mask is not None else "brick_mask"

show(
    [masks[name] * 255 for name in images],
    [f"{name}: mask ({mask_source[name]})" for name in images],
)

## Step 2: grey crop with contrast stretch, 96 × 96

The image becomes grey: the shape classifier only looks at the shape, colour is
a separate step of the pipeline. In grey, a red brick is almost black and a
yellow one almost white, while renders are mid-grey. So the grey values inside
the brick are spread over the full range 0..255 (the 2nd percentile becomes 0,
the 98th becomes 255), and everything outside the mask becomes one fixed grey
(128). Then a square around the brick (plus 8 % margin) is cut out and scaled
to 96 × 96 pixels.

After this step a brick 1×3 and a brick 1×4 look alike: the real size is lost.
On a sheet photo it comes back through the size prior (the brick is measured
in mm, `src/legorec/shape/size_prior.py`). Single images like these have no
known scale, so here the model has to do without it.

In [ ]:
crops = {}
for name, rgb in rgbs.items():
    height, width = masks[name].shape
    crops[name] = preprocess_crop(BrickCrop(rgb, masks[name], (0, 0, width, height)))
show(list(crops.values()), [f"{name}: model input" for name in crops])

# Grey values of the brick: before (full image, inside the mask) and
# after (crop without the grey background 128)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (name, image) in zip(axes, images.items()):
    before = np.asarray(image.convert("L"))[masks[name]]
    after = crops[name][crops[name] != 128]
    ax.hist(before, bins=64, range=(0, 255), density=True, alpha=0.6, label="before")
    ax.hist(after, bins=64, range=(0, 255), density=True, alpha=0.6, label="after")
    ax.set_title(f"{name}: grey values of the brick")
    ax.set_xlabel("grey value")
    ax.set_yticks([])
    ax.legend()
plt.show()

## Step 3: augmentation (only during training)

In training, every batch is changed at random: turned by any angle, zoomed and
shifted a little, made brighter or darker, with more or less contrast. So the
model has to learn the brick, not one exact picture. Below are eight random
versions of the render crop: this is what training batches look like.
Predictions (step 4) use the crop without changes.

In [ ]:
torch.manual_seed(0)  # the same random versions every time you run this cell

crop = torch.tensor(crops["render"])  # (96, 96), grey values 0..255
batch = normalize(crop.repeat(8, 1, 1, 1))  # 8 copies, shape (8, 1, 96, 96)
augmented = denormalize(augment_batch(batch)) * 255  # back to grey values

show([picture[0].numpy() for picture in augmented], [""] * 8, size=1.8)

## Step 4: prediction, top 3

`ShapeClassifier.predict_images` runs steps 1 and 2 exactly as above and
returns the three most likely parts. The model is the team default
(`models/shape.pt`, see the README). The green bar is the right part (3002).

In [ ]:
classifier = ShapeClassifier(MODEL_PATH)
names = classifier.names  # part number -> readable name


def plot_top3(ax, guesses, title):
    """Horizontal bars for [(part, probability), ...]; the right part is green."""
    guesses = guesses[::-1]  # matplotlib draws the first bar at the bottom
    labels = [names.get(part, part) for part, _ in guesses]
    probabilities = [probability for _, probability in guesses]
    colours = ["tab:green" if part == TRUE_PART else "tab:gray" for part, _ in guesses]
    bars = ax.barh(labels, probabilities, color=colours)
    ax.bar_label(bars, labels=[f"{p:.0%}" for p in probabilities], padding=3)
    ax.set_xlim(0, 1.2)
    ax.set_xticks([])
    ax.set_title(title)


top3 = {}  # image -> [(part, probability), ...]
for name, image in images.items():
    results, model_inputs = classifier.predict_images([image], k=3)
    assert np.array_equal(model_inputs[0], crops[name])  # the same as step 2
    top3[name] = results[0]

fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 2.5))
for ax, name in zip(axes, images):
    plot_top3(ax, top3[name], name)
plt.tight_layout()
plt.show()

## What we see

With the example images above and the team default `ok1tq8ks` (2026-11-21):

- **Render:** YOLO finds no brick on the black background (it was trained on
  our sheet photos), so `brick_mask` cuts it out. The model answers 3002 with
  64 %; second is 6215, a curved brick with the same 2×3 footprint.
- **Photo:** YOLO finds the brick, but its mask is rounder than the brick and
  cuts off a thin strip at the edge. The model answers 3002 with 76 %; second
  is 3001 (2×4, 19 %): the right kind of part in the wrong length, still the
  most common error type. On a sheet photo the size prior would push against
  it; here the scale is unknown.
- The older models of 2026-11-09 answered 3001 for this photo, with high
  confidence.

One example is not a measurement. Numbers on many images are in the
[experiment log](../docs/experiments.md).

**Try it yourself**

- The other renders in `data/raw/examples/` (poses `034L` and `049L`).
- One of your own photos: one brick, plain background, no strong shadow.
- A white brick on a light background: there YOLO often finds nothing and
  `brick_mask` takes over.

## Figure for the slides

All steps in one picture, for the team default model. It is saved as
`docs/figures/shape_walkthrough.png` (generated, not in Git).

In [ ]:
column_titles = ["input", "1 · find the brick", "2 · model input", "4 · top 3"]

with plt.rc_context({"font.size": 14}):
    fig, axes = plt.subplots(
        len(images),
        4,
        figsize=(16, 3.8 * len(images)),
        width_ratios=[1, 1, 1, 1.4],
        squeeze=False,
        layout="constrained",
    )
    for row, (name, image) in enumerate(images.items()):
        pictures = [np.asarray(image), masks[name] * 255, crops[name]]
        for column, picture in enumerate(pictures):
            axes[row, column].imshow(picture, cmap="gray", vmin=0, vmax=255)
            hide_ticks(axes[row, column])
        axes[row, 0].set_ylabel(name, fontsize=18)
        plot_top3(axes[row, 3], top3[name], title="")
    for ax, title in zip(axes[0], column_titles):
        ax.set_title(title)

FIGURES.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES / "shape_walkthrough.png", dpi=200, bbox_inches="tight")
plt.show()